In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
COUNTRY_FILE = RAW_DIR / "GlobalLandTemperaturesByCountry.csv"
GLOBAL_FILE = RAW_DIR / "Beginner_Climate_Change_Dataset_20_Features_1200_Rows.csv"

if not COUNTRY_FILE.exists() or not GLOBAL_FILE.exists():
    raise FileNotFoundError(
        f"Could not find input files in {RAW_DIR}. Check your folder structure and notebook working directory."
    )

In [2]:
df_country = pd.read_csv(COUNTRY_FILE)
df_global = pd.read_csv(GLOBAL_FILE)

In [3]:
df_country['dt'] = pd.to_datetime(df_country['dt'])
df_country['Year'] = df_country['dt'].dt.year
df_country['Month'] = df_country['dt'].dt.month
df_country['Country'] = df_country['Country'].astype(str).str.strip()

df_global = df_global.rename(columns={'year': 'Year', 'country': 'Country'})
df_global['Country'] = df_global['Country'].astype(str).str.strip()
df_global['Year'] = pd.to_numeric(df_global['Year'], errors='coerce')
df_country['Year'] = pd.to_numeric(df_country['Year'], errors='coerce')
df_country['AverageTemperature'] = pd.to_numeric(df_country['AverageTemperature'], errors='coerce')
df_country['AverageTemperatureUncertainty'] = pd.to_numeric(df_country['AverageTemperatureUncertainty'], errors='coerce')
value_cols = [c for c in df_global.columns if c not in ['Year', 'Country']]
df_global = df_global.groupby(['Year', 'Country'], as_index=False)[value_cols].mean(numeric_only=True)

df_country_yearly = df_country.groupby(['Year', 'Country'], as_index=False).agg(
    AverageTemperature=('AverageTemperature', 'mean'),
    AverageTemperatureUncertainty=('AverageTemperatureUncertainty', 'mean')
)

In [4]:
df_merged = pd.merge(
    df_country_yearly,
    df_global,
    on=['Year', 'Country'],
    how='inner',
    validate='one_to_one'
)

In [5]:
feature_cols = [c for c in df_global.columns if c not in ['Year', 'Country']]
df_model = df_merged.dropna(subset=['AverageTemperature'] + feature_cols)
df_model = df_model.sort_values(['Country', 'Year']).reset_index(drop=True)

In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_file = PROCESSED_DIR / "model_dataset.csv"
df_model.to_csv(output_file, index=False)
print(f"Saved: {output_file}")

Saved: c:\Users\bulba\OneDrive\Desktop\Climate-Change-Predictor\data\processed\model_dataset.csv
